# Cluster analysis with K-means

Algorisme de l'article [Spectral-clustering approach to Lagrangian vortex detection](https://arxiv.org/pdf/1506.02258) a partir de trajectòries del sistema dinàmic donat 
pel següent sistema d'EDOs:  $x'=y + \epsilon f(t)$,  $y'=x-x^3$, on $f(t)=sin(t)$.

In [ ]:
import sys
sys.path.append("..")
from src import *
import numpy as np
np.set_printoptions(precision=3, suppress=True)

%load_ext autoreload
%autoreload 2

##### Paràmetres

In [ ]:
epsilon = 0
funcio_soroll = np.sin
dimensio = 2
t_span = (0, 4*np.pi)
t_steps = 300
t_valors = np.linspace(t_span[0], t_span[1], t_steps)
x_min, x_max = (-1.6, 1.6)
y_min, y_max = (-1, 1)
espai_entre_punts = 0.05
constant_diagonal = 100000
max_clusters = 10

### 1. Generar $n$ posicions inicials aleatòries

In [ ]:
condicions_inicials = generar_condicions_inicials(
    espai_entre_punts, (x_min, x_max), (y_min, y_max)
)
num_trajectories = len(condicions_inicials)

In [ ]:
# grafica_punts(condicions_inicials, dibuixa_regions=False)

### 2. Generar $n$ trajectòries, una per a cada posició inicial

In [ ]:
trajectories = generar_trajectories(edo_duffing, condicions_inicials, t_span, t_valors)
print("(Num trajectories, t_steps, dimensio) =", trajectories.shape)

In [ ]:
# grafica_trajectories(trajectories, dibuixa_regions=False)

In [ ]:
exemples = np.array((trajectories[0], trajectories[1165], trajectories[1323]))
# grafica_trajectories(exemples)

### 3. Calcular distàncies $r_{ij}$ entre trajectòries i crear graf $G = (V, E, W)$

Els vèrtexs són cadascuna de les $n$ trajectòries: $V=\{v_1,...,v_n\}$.

Cada aresta $e_{ij}\in E\subseteq V\times V$ està associada al pes $w_{ij}\in W\in\R^{n\times n}$, on $W$ és la matriu de similaritat i $w_{ij} = 1/r_{ij}$. Cada pes representa quant de properes són dues trajectòries ($0 \rightarrow$ poc, i $\infty \rightarrow$ molt).

### 4 Matriu de similaritat

In [ ]:
matriu_pesos = calcula_matriu_pesos(trajectories)

In [ ]:
# imprimeix_estadistics(matriu_pesos)

##### 4.1 Opció A: triar radi d'esparsificació tal que el 95% de la matriu de pesos es torni nul·la

In [ ]:
matriu_similaritat_W, sparsification_tol, sparsification_percent = \
    sparcify(matriu_pesos, percent=95)
print(f"S'ha obtingut una esparsificació del "
      f"{sparsification_percent*100:.0f}% usant una tolerància de "
      f"{sparsification_tol:.3f}")
np.fill_diagonal(matriu_similaritat_W, constant_diagonal)
# print(matriu_similaritat_W)

##### Opció B: triar radi d'esparsificació que maximitza la diferència màxima entre VAPs consecutius

In [ ]:
diffs, nums_clusters, radis, estadistics = calcula_diffs_vs_radis(matriu_pesos, constant_diagonal)

In [ ]:
grafica_dif_vs_radi(diffs, nums_clusters, radis, estadistics)

### 5. Calcular la matriu diagonal de graus $D$, on $D_{ii}=\sum _{j=0}^{n} w_{ij}$

In [ ]:
matriu_grau_D = calcula_matriu_grau(matriu_similaritat_W)

### 6. Descomposició espectral: $Lu =\lambda Du$

In [ ]:
vaps, veps = calcula_vaps(matriu_grau_D, matriu_similaritat_W, max_clusters)
print("vaps =", vaps)
print("veps.shape =", veps.shape)

### 7. Estimació del nombre de clústers $k$ = argmin [max($g_i$)]

In [ ]:
print("nombre de clusters =", num_clusters := calcula_num_clusters(vaps))

### 8. $k$-means

In [ ]:
labels = troba_clusters(num_clusters, veps)

In [ ]:
grafica_clusters(condicions_inicials, labels, num_clusters, sparsification_tol, sparsification_percent, t_steps, t_span)